In [ ]:
%run fabric-notebook-config

In [ ]:
# notebook values
domain_config_column_names = ['Domain ID', 'Domain Name', 'Description', 'Parent Domain ID', 'Contributors Scope']
domain_workspace_column_names = ['Domain ID', "Workspace ID"]
domain_structure_file = 'domain-intended.csv'
domain_workspace_mapping_file = 'workspace-domain-intended.csv'

domain_structure_path = f"{lakehouse_path}/{domain_structure_file}"
domain_workspace_mapping_path = f"{lakehouse_path}/{domain_workspace_mapping_file}"

### Current state (API)

In [ ]:
# get a unique list of Domains to iterate over
with labs.service_principal_authentication(
        key_vault_uri = key_vault_uri,
        key_vault_tenant_id = 'tenant-fabric-sp',
        key_vault_client_id = 'fabric-admin-sp-app-id', 
        key_vault_client_secret = admin_sp_client_secret_name[1]
    ):
    df_domains_current = labs.admin.list_domains()
domain_list = set(df_domains_current['Domain ID'])

### Target state (Lakehouse)

In [ ]:
# Load domain_structure_file into pandas DataFrame
domain_intended_spark = spark.read.format("csv").option('header', 'true').load(domain_structure_path)
domain_intended = domain_intended_spark.toPandas()
# prep to enforce admins only
domain_intended['Contributors Scope'] = 'AdminsOnly'

In [ ]:
# Load mapping file into pandas DataFrame
workspace_domain_spark = spark.read.format("csv").option('header', 'true').load(domain_workspace_mapping_path)
workspace_domain_intended = workspace_domain_spark.toPandas()

workspace_domain_intended.columns = domain_workspace_column_names
workspace_domain_intended.sort_values(by = domain_workspace_column_names, inplace=True)

## Function definitions

In [ ]:
def check_domain_validity(input_data_frame):
    """
    Validates domain names and descriptions based on length constraints and flags invalid entries.

    Parameters
    ----------
    input_data_frame : pandas.DataFrame
        A DataFrame containing domain information with columns `Domain Name` and `Description`.

    Returns
    -------
    pandas.DataFrame or None
        Returns a DataFrame containing rows flagged as invalid if any issues are found. 
        If all domains are valid, returns None.

    Notes
    -----
    - Domain names exceeding 40 characters are flagged as invalid.
    - Domain descriptions exceeding 256 characters are flagged as invalid.
    - Adds a new column `domain_invalid` to the input DataFrame to indicate validity.

    """

    assessment_df = input_data_frame.copy()

    # check if name too long
    domain_name_len_valid = np.invert(np.array(list(map(len, assessment_df['Domain Name']))) > 40)
    # check if desc too long
    domain_desc_len_valid = np.invert(np.array(list(map(len, assessment_df['Description']))) > 256)
    # mush the dataframes together
    domain_check_valid = np.logical_and(domain_name_len_valid, domain_desc_len_valid)
    assessment_df.insert(loc=0, column = "domain_valid", value = domain_check_valid)

    # begin logical handling of stuff that'll throw errors
    # start is fine
    if all(domain_check_valid):
        print('Domain validity: No issues')
        return None
    # start is problematic
    else:
        if all(domain_name_len_valid):
            print("Names are valid")
        else: 
            print("Domain validity: Some names are too long")
        if all(domain_desc_len_valid):
            print("Descriptions are valid")
        else:
            print("Domain validity: Some descriptions are too long")
        # return problematic rows
        return assessment_df[internal_df_copy['domain_valid'] == False]

In [ ]:
def check_domain_workspaces_still_assigned(input_df):
    """
    Checks if there are any workspaces still assigned to the specified domains.

    Parameters
    ----------
    input_df : pandas.DataFrame
        A DataFrame containing domain information with at least a `Domain ID` column.

    Returns
    -------
    pandas.DataFrame
        A DataFrame listing workspaces still assigned to the domains, including columns:
        `Domain ID`, `Workspace ID`, and `Workspace Name`.

    Notes
    -----
    - Uses the `labs.admin.list_domain_workspaces` function to retrieve workspace assignments for each domain.
    - If no workspaces are assigned, returns an empty DataFrame.

    Examples
    --------
    >>> import pandas as pd
    >>> data = {'Domain ID': ['domain1', 'domain2']}
    >>> df = pd.DataFrame(data)
    >>> check_domain_workspaces_still_assigned(df)
    """
    # set up empty array
    workspaces_still_assigned = pd.DataFrame(columns = ['Domain ID', 'Workspace ID', 'Workspace Name'])
    # loop through a still assigned workspace check
    with labs.service_principal_authentication(
        key_vault_uri = key_vault_uri,
        key_vault_tenant_id = 'tenant-fabric-sp',
        key_vault_client_id = 'fabric-admin-sp-app-id', 
        key_vault_client_secret = admin_sp_client_secret_name[1]
    ):
        for domain, row in input_df.iterrows():
        
            list_ws_df = labs.admin.list_domain_workspaces(row['Domain ID'])
        list_ws_df.insert(loc=0, column = "Domain ID", value = row['Domain ID'])
        # iteratively added to the df
        workspaces_still_assigned = pd.concat([workspaces_still_assigned, list_ws_df])
    return workspaces_still_assigned

In [ ]:
def check_empty_domains():
    '''
    Checks for empty domains (domains with no assigned workspaces) and returns a DataFrame of such domains.

    Parameters
    ----------
    None

    Returns
    -------
    pandas.DataFrame or None
        - Returns a DataFrame containing details of empty domains if any exist.
        - Returns None if all domains have assigned workspaces.

    Notes
    -----
    - Queries the current list of domains using `labs.admin.list_domains`.
    - For each domain, counts the number of workspaces assigned using `labs.admin.list_domain_workspaces`.
    - Adds a new column `workspace count` to the domains DataFrame to indicate the number of workspaces assigned to each domain.
    - If empty domains exist, prints a message and returns a DataFrame of these domains for review.
    - If no empty domains exist, prints a message and returns None.
    
    '''
    # query the current domains (post deletion) and check if any are empty
    with labs.service_principal_authentication(
        key_vault_uri = key_vault_uri,
        key_vault_tenant_id = 'tenant-fabric-sp',
        key_vault_client_id = 'fabric-admin-sp-app-id', 
        key_vault_client_secret = admin_sp_client_secret_name[1]
        ):
        df_domains_current = labs.admin.list_domains()
        domain_workspace_count = []
        # for each Domain, count how many Workspaces there are
        for domain, row in df_domains_current.iterrows():
            workspaces_assigned = labs.admin.list_domain_workspaces(row['Domain ID'])
            domain_workspace_count.append(workspaces_assigned.shape[0])
        # add that list to the dataframe
        df_domains_current.insert(loc=df_domains_current.shape[1], column = "workspace count", value = domain_workspace_count)
        # return DF or empty domains
        df_domains_empty = df_domains_current[df_domains_current['workspace count'] == 0]
        if df_domains_empty.shape[0] == 0:
            print("Nothing to clean up, all Domains have assigned Workspaces!!")
            return None
        else: 
            print("Some empty Domains exist, and this function's return contains these Domains.\nReview if these Domains are still required, or if they can be deleted. If Deleting, remove them from the Domain config file, and the associated scripts will do the clean up")
            return df_domains_empty 

In [ ]:
def add_domains_refuse(input_df):
    """
    Flags that new domains cannot be created within the current flow and must be added elsewhere.

    Parameters
    ----------
    input_df : pandas.DataFrame
        A DataFrame containing Semantic link input specification of domains to be added.

    Returns
    -------
    None
        Exits the process with an error message if there are domains to be added. 
        Returns 'None' if no addition requests exist.

    Notes
    -----
    - If domains are flagged for addition, exits the process with a message and invalid domains.
    - If no domains are flagged for addition, prints a message indicating no addition requests.
    """

    domain_new_dict = {
    "message": "Cannot create new domains here, run a script referencing notebook `fabric-domain-create-util`, then add the IDs to the config file!",
    "new_domains_invalid": to_add
    }
    # if items in list, stop proceeding
    if input_df.shape[0] != 0:
        print(domain_new_dict['message'])
        mssparkutils.notebook.exit(domain_new_dict)
    else:
        print("No addition requests")
        return None

In [ ]:
def update_domains(input_df):
    """
    Function which updates the title an descriptions of existing domains - useful for rebrands or sub-assignments / re-allocations
    Checks if the title or the desc is wrong, and revises where relevant
    

    This may need to change in the future, as the API which this function calls is being depreciated 2026-05,
    - the new one doesn't have contributor scopes
    - new one does allow for renames - which this currently doesn't handle
    Parameters
    ----------
    input_df : pandas.DataFrame
        A DataFrame containing domain information with at least `Domain ID` and `Description_x` columns.

    Returns
    -------
    str or None
        Returns "Complete" if domains are successfully updated. 
        Returns 'None' if there are no domains to update.

    Notes
    -----
    - Uses the `labs.admin.update_domain` function to update domain descriptions.
    - The API used by this function is set to be deprecated in May 2026. Future updates may require changes to this function.

    """

    if input_df.shape[0] == 0:
        print('nothing to update')
        return None
    else:
        print("Updating domains")
        for domain, row in input_df.iterrows():
            with labs.service_principal_authentication(
                key_vault_uri = key_vault_uri,
                key_vault_tenant_id = 'tenant-fabric-sp',
                key_vault_client_id = 'fabric-admin-sp-app-id', 
                key_vault_client_secret = admin_sp_client_secret_name[1]
            ):
                labs.admin.update_domain(row['Domain ID'], row['Description_x'], "AdminsOnly")
        print("Complete")
        return "Complete"

In [ ]:
def delete_domains(input_df):
    """
    Deletes domains that are not in the desired configuration file and which have no workspaces assigned.

    Parameters
    ----------
    input_df : pandas.DataFrame
        A DataFrame containing domain information with at least `Domain ID` and `Domain Name_y` columns.

    Returns
    -------
    str or None
        Returns "Complete" if domains are successfully deleted. Returns None if there are no domains to delete.

    Notes
    -----
    - Checks if any workspaces are still assigned to the domains using `check_domain_workspaces_still_assigned`.
    - If workspaces are assigned, exits the process with a dictionary format error message and invalid domains list.
    - Deletes domains using the `labs.admin.delete_domain` function.

    Examples
    --------
    >>> import pandas as pd
    >>> data = {'Domain ID': ['domain1', 'domain2'], 'Domain Name': ['Domain1', 'Domain2']}
    >>> df = pd.DataFrame(data)
    >>> delete_domains(df)
    """

    if input_df.shape[0] == 0:
        print("nothing to delete")
        return None
    else:
        print("checking if there are any workspaces still assigned")
        # check for any workspaces which are still assigned to domains scheduled for deletion
        domain_workspaces_still_assigned = check_domain_workspaces_still_assigned(input_df)
        # make a useful error value
        domain_still_workspaces_dict = {
        "message": "You still have workspaces assigned to the domains!",
        "invalid_domains": domain_workspaces_still_assigned
            }
            # if items in list, stop proceeding
        if domain_workspaces_still_assigned.shape[0] != 0:
            print("Still stuff assigned!")
            mssparkutils.notebook.exit(domain_still_workspaces_dict)
        # otherwise get deleting
        else: 
            for domain, row in input_df.iterrows():
                print("Deleting domain: ", row['Domain Name'])
                with labs.service_principal_authentication(
                    key_vault_uri = key_vault_uri,
                    key_vault_tenant_id = 'tenant-fabric-sp',
                    key_vault_client_id = 'fabric-admin-sp-app-id', 
                    key_vault_client_secret = admin_sp_client_secret_name[1]
                ):
                    labs.admin.delete_domain(row['Domain ID'])
            print("All deletions completed!\nPlease ensure that any corresponding records have also been removed from the Domain Config file, so they're not believed to be 'target state'")
            return "Complete"

In [ ]:
def domain_workspace_detach(df_to_remove):
    '''
    Detaches workspaces from their current domain based on the provided DataFrame.

    Parameters
    ----------
    df_to_remove : pandas.DataFrame
        A DataFrame containing information about workspaces that need to be completely removed from their current domain. 
        It must include columns such as `Domain ID_y` and `Workspace ID`.

    Returns
    -------
    str
        Returns None if there are no workspaces to detach.
        Returns "end removal logic" after successfully detaching the workspaces.

    Notes
    -----
    - Uses the `labs.admin.unassign_domain_workspaces` function from Semantic Link Labs to detach workspaces.
    - The function retrieves the workspace name using `labs.admin.list_workspace_access_details` before detaching.
    - If the DataFrame is empty, no operations are performed.

    '''
    print("checking if any workspaces are slated for removal from domains completely")
    # do we need to completely de-assign any workspaces from their current state Domain? (likely bc not in the config file) 
    if pd.DataFrame(df_to_remove).size == 0:
        print("no removals needed")
        return None
    else:
        # removal logic to go here
        for index, row in df_to_remove.iterrows():
            with labs.service_principal_authentication(
                key_vault_uri = key_vault_uri,
                key_vault_tenant_id = 'tenant-fabric-sp',
                key_vault_client_id = 'fabric-admin-sp-app-id', 
                key_vault_client_secret = admin_sp_client_secret_name[1]
            ):
                workspace_name = labs.admin.list_workspace_access_details(row['Workspace ID'])['Workspace Name'].iloc[0]
                labs.admin.unassign_domain_workspaces(
                row['Domain ID_y'],
                workspace_name
            )
        print('end removal logic')
        return None

In [ ]:
def domain_workspace_assign(df_to_assign):
    '''
     Assigns new workspaces to a domain based on the provided DataFrame.

    Parameters
    ----------
    df_to_assign : pandas.DataFrame
        A DataFrame containing information about new workspaces that need to be assigned to a domain. 
        It must include columns such as `Domain ID` and `Workspace ID`.

    Returns
    -------
    str
        Returns None if there are no new workspaces to assign.
        Returns "end add logic" after successfully assigning the workspaces.

    Notes
    -----
    - Uses the `labs.admin.assign_domain_workspaces` function from Semantic Link Labs to assign workspaces to domains.
    - If the DataFrame is empty, no operations are performed.

    '''
    print("Checking if any workspaces are new, and need to be assigned")
    # do we need to add any workspaces?
    if pd.DataFrame(df_to_assign).size == 0: 
        print("no additions needed")
        return None
    else:
        for index, row in df_to_assign.iterrows():
            with labs.service_principal_authentication(
                key_vault_uri = key_vault_uri,
                key_vault_tenant_id = 'tenant-fabric-sp',
                key_vault_client_id = 'fabric-admin-sp-app-id', 
                key_vault_client_secret = admin_sp_client_secret_name[1]
            ):
                labs.admin.assign_domain_workspaces(row['Domain ID'], row['Workspace ID'])
        return('end add logic')

In [ ]:
def domain_workspace_move(df_to_move):
    '''
    Reassigns workspaces to a new domain based on the provided DataFrame.

    Parameters
    ----------
    df_to_move : pandas.DataFrame
        A DataFrame containing information about workspaces that need to be reassigned to a new domain. 
        It must include columns such as `Domain ID_x` and `Workspace ID`.

    Returns
    -------
    str
        Returns None if there are no workspaces to reassign
        Returns "end relocations logic" after successfully reassigning the workspaces.

    Notes
    -----
    - Uses the `labs.admin.assign_domain_workspaces` function from Semantic Link Labs to reassign workspaces.
    - If the DataFrame is empty, no operations are performed.

    '''
    print("Checking if any Workspaces need to be re-assigned")
    if pd.DataFrame(df_to_move).size == 0:
        print("no relocations needed")
        return None
    else:
        # removal logic to go here
        with labs.service_principal_authentication(
            key_vault_uri = key_vault_uri,
            key_vault_tenant_id = 'tenant-fabric-sp',
            key_vault_client_id = 'fabric-admin-sp-app-id', 
            key_vault_client_secret = admin_sp_client_secret_name[1]
        ):
            for index, row in df_to_move.iterrows():
                labs.admin.assign_domain_workspaces(row['Domain ID_x'], row['Workspace ID'])
            return('end relocations logic')